In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/11 20:50:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../data/25.06/"
path_to_intermediate_data_folder = "../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence")


efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)


g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")
g_p_s.count()

26/08/11 20:50:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

In [ ]:
x = all_evidence.select("targetId", "diseaseId", "sourceId").distinct().groupBy("sourceId").count()

In [7]:
x.show(100)

+------------------+-------+
|          sourceId|  count|
+------------------+-------+
|gwas_credible_sets| 551409|
|         europepmc|2373042|
|            crispr|    517|
|  uniprot_variants|   5142|
|  genomics_england|  10523|
|       gene_burden|   7860|
|            chembl|  74187|
|        slapenrich|  22939|
|     crispr_screen|  10980|
|  expression_atlas| 160057|
|           intogen|   2596|
|          reactome|   2482|
|          orphanet|   6288|
|            sysbio|    336|
| cancer_biomarkers|    468|
|cancer_gene_census|  82754|
|           progeny|    378|
|           clingen|   3047|
|               eva|  39302|
|    gene2phenotype|   3219|
|uniprot_literature|   6636|
|       eva_somatic|   2871|
|              impc| 662277|
+------------------+-------+



In [ ]:
x = x.toPandas()

AttributeError: 'function' object has no attribute 'toPandas'

# Version 2


In [23]:
x = session.spark.read.csv("combined_ti_TA.tsv", sep="\t", header=True)

In [24]:
x.show(1)

+-----------+----------+------+-----------+----+------------------+--------------------+---------+----+---------+-------+-------+-------+--------+--------+--------+--------+------+-----------+----+-------------+--------------------+------------+----------+------------------+--------------------+----------+------------+------+-----------------+---------+--------+---------+-----------+----------+--------------------------------+---------------+------------------+
|        uid|similarity|catnum|     ti_uid|gene|indication_mesh_id|indication_mesh_term|     hcat|acat|     ccat|hcatnum|acatnum|ccatnum|succ_p_1|succ_1_2|succ_2_3|succ_3_a|orphan|year_launch|arow|assoc_mesh_id|     assoc_mesh_term|assoc_source|assoc_info|    original_trait|       original_link|assoc_year|pic_qtl_pval|pic_h4|    af_gnomad_nfe|l2g_share|l2g_rank|      cat|assoc_share|assoc_rank|highest_phase_with_known_outcome|genetic_insight|     target_status|
+-----------+----------+------+-----------+----+------------------+-

In [25]:
x = x.withColumnRenamed("gene", "approvedSymbol")

In [26]:
target = session.spark.read.parquet(path_to_release_folder + "output/target")

In [27]:
x.count()

25713

In [28]:
target = target.select("id", "approvedSymbol").withColumnRenamed("id", "targetId")
target_agg = target.groupBy("approvedSymbol").agg(f.collect_list("targetId").alias("targetIds"))
x = x.join(target_agg, on=["approvedSymbol"], how="left").cache()
x.count()

25713

In [29]:
x.show(1)

+--------------+-----------+----------+------+-----------+------------------+--------------------+---------+----+---------+-------+-------+-------+--------+--------+--------+--------+------+-----------+----+-------------+--------------------+------------+----------+------------------+--------------------+----------+------------+------+-----------------+---------+--------+---------+-----------+----------+--------------------------------+---------------+------------------+-----------------+
|approvedSymbol|        uid|similarity|catnum|     ti_uid|indication_mesh_id|indication_mesh_term|     hcat|acat|     ccat|hcatnum|acatnum|ccatnum|succ_p_1|succ_1_2|succ_2_3|succ_3_a|orphan|year_launch|arow|assoc_mesh_id|     assoc_mesh_term|assoc_source|assoc_info|    original_trait|       original_link|assoc_year|pic_qtl_pval|pic_h4|    af_gnomad_nfe|l2g_share|l2g_rank|      cat|assoc_share|assoc_rank|highest_phase_with_known_outcome|genetic_insight|     target_status|        targetIds|
+-----------

In [30]:
x.filter(f.col("targetIds").isNull()).count()

95

In [31]:
df_exploded = disease_index_orig.withColumn("dbXRef", f.explode(f.col("dbXRefs"))).withColumn(
    "dbXRef", f.upper(f.col("dbXRef"))
)
mesh_pattern = r"(?i)MESH:\s*([^,\s]+)"

df_mesh = df_exploded.withColumn("meshId", f.regexp_extract(f.col("dbXRef"), mesh_pattern, 1))
df_mesh = df_mesh.select("id", "meshId").filter(f.col("meshId") != "").distinct()
df_mesh.count()

df_exploded2 = disease_index_orig.withColumn("obsoleteXRef", f.explode(f.col("obsoleteXRefs"))).withColumn(
    "obsoleteXRef", f.upper(f.col("obsoleteXRef"))
)

mesh_pattern = r"(?i)MESH:\s*([^,\s]+)"

df_mesh2 = df_exploded2.withColumn("meshId", f.regexp_extract(f.col("obsoleteXRef"), mesh_pattern, 1))
df_mesh2 = df_mesh2.select("id", "meshId").filter(f.col("meshId") != "").distinct()
df_mesh2.count()

df_mesh_all = df_mesh.unionByName(df_mesh2).distinct()
df_mesh_all.count()

6695

In [32]:
df_mesh_agg = df_mesh_all.groupBy("meshId").agg(f.collect_list("id").alias("diseaseIds"))
df_mesh_agg.count()

6207

In [33]:
x = x.withColumnRenamed("indication_mesh_id", "meshId")

In [34]:
x.count()

25713

In [35]:
x = x.join(df_mesh_agg.select("diseaseIds", "meshId"), on=["meshId"], how="left").cache()
x.count()

25713

In [36]:
x.show(1)

+-------+--------------+-----------+----------+------+-----------+--------------------+---------+----+---------+-------+-------+-------+--------+--------+--------+--------+------+-----------+----+-------------+--------------------+------------+----------+------------------+--------------------+----------+------------+------+-----------------+---------+--------+---------+-----------+----------+--------------------------------+---------------+------------------+-----------------+--------------------+
| meshId|approvedSymbol|        uid|similarity|catnum|     ti_uid|indication_mesh_term|     hcat|acat|     ccat|hcatnum|acatnum|ccatnum|succ_p_1|succ_1_2|succ_2_3|succ_3_a|orphan|year_launch|arow|assoc_mesh_id|     assoc_mesh_term|assoc_source|assoc_info|    original_trait|       original_link|assoc_year|pic_qtl_pval|pic_h4|    af_gnomad_nfe|l2g_share|l2g_rank|      cat|assoc_share|assoc_rank|highest_phase_with_known_outcome|genetic_insight|     target_status|        targetIds|          dise

In [37]:
x.filter(f.col("diseaseIds").isNull()).count()

2523

In [38]:
x.count()

25713

In [39]:
x_exp = x.withColumn("diseaseId", f.explode(f.col("diseaseIds"))).withColumn("targetId", f.explode(f.col("targetIds")))
x_exp.count()

26455

In [40]:
x_exp = x_exp.filter(f.col("diseaseId").isNotNull() & f.col("targetId").isNotNull())
x_exp.count()

26455

In [41]:
x_exp = x_exp.withColumn("outcome", f.when(f.col("ccatnum") == 5, 1).otherwise(0))

In [42]:
x_exp = x_exp.withColumn(
    "geneticSupport_old", f.when(f.col("target_status") == "genetically supported target", 1).otherwise(0)
)

In [67]:
x_exp.filter(f.col("geneticSupport_old") == 1).groupBy("genetic_insight").count().show()

+---------------+-----+
|genetic_insight|count|
+---------------+-----+
|           gwas|  555|
|           both| 1609|
|      omim/tcga|   81|
+---------------+-----+



26/08/12 01:05:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 979915 ms exceeds timeout 120000 ms
26/08/12 01:05:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/08/12 01:05:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [43]:
import pandas as pd

x_exp_pd = x_exp.filter(~f.col("diseaseId").isin(efo_to_remove))
x_exp_pd = x_exp_pd.filter(f.col("ccatnum") >= 2)

In [44]:
full_l2g_evidnce = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)
full_l2g_evidnce.count()

77071

In [45]:
evid_indirect = chemblDrugEnrichment.evidence_to_indirect_assosiations(
    full_l2g_evidnce,
    disease_index_orig,
    use_max=True,
    efo_to_remove=efo_to_remove,
).cache()
evid_indirect.count()

26/08/11 21:06:09 WARN CacheManager: Asked to cache already cached data.


151704

In [46]:
evid_indirect = (
    evid_indirect.select("targetId", "diseaseId")
    .withColumnRenamed("targetId", "geneId")
    .withColumn("geneticSupport", f.lit(1))
)

In [47]:
x_exp_pd = (
    x_exp_pd.join(evid_indirect.withColumnRenamed("geneId", "targetId"), ["targetId", "diseaseId"], "left")
    .fillna({"geneticSupport": 0})
    .cache()
)
x_exp_pd.count()

7390

In [48]:
x_exp_pd.show(1)

+---------------+-------------+-------+--------------+-----------+----------+------+-----------+--------------------+---------+----+---------+-------+-------+-------+--------+--------+--------+--------+------+-----------+----+-------------+--------------------+------------+----------+------------------+--------------------+----------+------------+------+-----------------+---------+--------+---------+-----------+----------+--------------------------------+---------------+------------------+-----------------+--------------------+-------+------------------+--------------+
|       targetId|    diseaseId| meshId|approvedSymbol|        uid|similarity|catnum|     ti_uid|indication_mesh_term|     hcat|acat|     ccat|hcatnum|acatnum|ccatnum|succ_p_1|succ_1_2|succ_2_3|succ_3_a|orphan|year_launch|arow|assoc_mesh_id|     assoc_mesh_term|assoc_source|assoc_info|    original_trait|       original_link|assoc_year|pic_qtl_pval|pic_h4|    af_gnomad_nfe|l2g_share|l2g_rank|      cat|assoc_share|assoc_ran

In [49]:
x_exp_pd = (
    x_exp_pd.join(
        g_p_s.select("geneId", "uniqueTherapeuticAreas", "uniqueDiseases").withColumnRenamed("geneId", "targetId"),
        "targetId",
        "left",
    )
    .fillna({"uniqueTherapeuticAreas": 0, "uniqueDiseases": 0})
    .cache()
)
x_exp_pd.count()

7390

In [50]:
x_exp_pd.show(1)

+---------------+-------------+-------+--------------+-----------+----------+------+-----------+--------------------+---------+----+---------+-------+-------+-------+--------+--------+--------+--------+------+-----------+----+-------------+--------------------+------------+----------+------------------+--------------------+----------+------------+------+-----------------+---------+--------+---------+-----------+----------+--------------------------------+---------------+------------------+-----------------+--------------------+-------+------------------+--------------+----------------------+--------------+
|       targetId|    diseaseId| meshId|approvedSymbol|        uid|similarity|catnum|     ti_uid|indication_mesh_term|     hcat|acat|     ccat|hcatnum|acatnum|ccatnum|succ_p_1|succ_1_2|succ_2_3|succ_3_a|orphan|year_launch|arow|assoc_mesh_id|     assoc_mesh_term|assoc_source|assoc_info|    original_trait|       original_link|assoc_year|pic_qtl_pval|pic_h4|    af_gnomad_nfe|l2g_share|l2

In [51]:
x_exp_pd.groupBy("ccatnum").count().show()

+-------+-----+
|ccatnum|count|
+-------+-----+
|      3| 3303|
|      5|  913|
|      4|  900|
|      2| 2274|
+-------+-----+



In [52]:
x_exp_pd = x_exp_pd.withColumn("maxClinicalPhase", f.col("ccatnum").cast("int") - 1)

In [53]:
x_exp_pd.groupBy("maxClinicalPhase").count().show()

+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|               1| 2274|
|               3|  900|
|               4|  913|
|               2| 3303|
+----------------+-----+



In [61]:
import statsmodels.formula.api as smf

df = x_exp_pd.toPandas()
model = smf.logit("outcome ~ geneticSupport_old", data=df).fit(disp=False)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                 7390
Model:                          Logit   Df Residuals:                     7388
Method:                           MLE   Df Model:                            1
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:                 0.01396
Time:                        21:19:03   Log-Likelihood:                -2724.8
converged:                       True   LL-Null:                       -2763.3
Covariance Type:            nonrobust   LLR p-value:                 1.592e-18
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.0869      0.039    -52.852      0.000      -2.164      -2.009
geneticSupport_old     0.8431      0.091      9.271      0.000       0.665       1.021


In [62]:
import statsmodels.formula.api as smf

df = x_exp_pd.toPandas()
model = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                 7390
Model:                          Logit   Df Residuals:                     7388
Method:                           MLE   Df Model:                            1
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:                0.002683
Time:                        21:19:05   Log-Likelihood:                -2755.9
converged:                       True   LL-Null:                       -2763.3
Covariance Type:            nonrobust   LLR p-value:                 0.0001178
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.9973      0.037    -53.896      0.000      -2.070      -1.925
geneticSupport     0.5036      0.125      4.030      0.000       0.259       0.748


In [ ]:
np.exp(0.8431)

np.float64(2.3235588560625033)

In [ ]:
np.exp(0.5036)

np.float64(1.654667363820486)

In [ ]:
x_exp_pd.toPandas().to_csv(path_to_intermediate_data_folder + "minikel_etal_processed_data_v2.csv", index=False)

In [ ]:
x_exp_pd.filter(f.col("geneticSupport") == 1).count()

469

In [ ]:
x_exp_pd.filter(f.col("geneticSupport_old") == 1).count()

858

# Load


In [4]:
x = session.spark.read.csv("drug_phase_summary.tsv", sep="\t", header=True)

In [5]:
x.show(1)

+-----------+--------+---------+-----------+---------+-------+
|     ti_uid|phasenum|    phase|maxphasenum| maxphase|n_drugs|
+-----------+--------+---------+-----------+---------+-------+
|A2M-D002545|       5|Phase III|          5|Phase III|      1|
+-----------+--------+---------+-----------+---------+-------+
only showing top 1 row



In [ ]:
# x = x.withColumn("approvedSymbol", f.split(x["ti_uid"], "-").getItem(0)).withColumn(
#    "meshId", f.split(x["ti_uid"], "-").getItem(1)
# )

In [ ]:
x = x.withColumn("approvedSymbol", f.regexp_extract(f.col("ti_uid"), r"(.*)-([^-]+)$", 1)).withColumn(
    "meshId", f.regexp_extract(f.col("ti_uid"), r"(.*)-([^-]+)$", 2)
)

In [8]:
x = x.withColumn("approvedSymbol", f.upper(x["approvedSymbol"])).withColumn("meshId", f.upper(x["meshId"]))

In [9]:
x.show()

+-------------+--------+-----------+-----------+-----------+-------+--------------+-------+
|       ti_uid|phasenum|      phase|maxphasenum|   maxphase|n_drugs|approvedSymbol| meshId|
+-------------+--------+-----------+-----------+-----------+-------+--------------+-------+
|  A2M-D002545|       5|  Phase III|          5|  Phase III|      1|           A2M|D002545|
|  A2M-D007511|       5|  Phase III|          5|  Phase III|      1|           A2M|D007511|
|  A2M-D010003|       2|Preclinical|          2|Preclinical|      1|           A2M|D010003|
|  A2M-D016491|       5|  Phase III|          5|  Phase III|      1|           A2M|D016491|
|  A2M-D020521|       5|  Phase III|          5|  Phase III|      1|           A2M|D020521|
|  A2M-D023903|       5|  Phase III|          5|  Phase III|      1|           A2M|D023903|
|AADAT-D012559|       2|Preclinical|          2|Preclinical|      3|         AADAT|D012559|
| AAK1-D009437|       2|Preclinical|          2|Preclinical|      1|          AA

In [10]:
x.count()

46015

In [ ]:
x.filter(f.col("ti_uid") == "ERVW-1-C000711409").show()

+-----------------+--------+-----------+-----------+--------+-------+--------------+----------+
|           ti_uid|phasenum|      phase|maxphasenum|maxphase|n_drugs|approvedSymbol|    meshId|
+-----------------+--------+-----------+-----------+--------+-------+--------------+----------+
|ERVW-1-C000711409|       2|Preclinical|          4|Phase II|      1|        ERVW-1|C000711409|
+-----------------+--------+-----------+-----------+--------+-------+--------------+----------+



# Combine with the target


In [12]:
target = session.spark.read.parquet(path_to_release_folder + "output/target")

In [13]:
target.show(1)

+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+----------------+--------------------+--------------------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------+---------+
|             id|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes| approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|targetClass| obsoleteSymbols|       obsoleteNames|          constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|          homologues|        tractability|safetyLiabilitie

In [14]:
target = target.select("id", "approvedSymbol").withColumnRenamed("id", "targetId")

In [15]:
target.count()

78726

In [16]:
target.select("approvedSymbol").distinct().count()

77111

In [17]:
target.show(1)

+---------------+--------------+
|       targetId|approvedSymbol|
+---------------+--------------+
|ENSG00000000003|        TSPAN6|
+---------------+--------------+
only showing top 1 row



In [18]:
target_agg = target.groupBy("approvedSymbol").agg(f.collect_list("targetId").alias("targetIds"))

In [19]:
target_agg.show(1)

+--------------+--------------------+
|approvedSymbol|           targetIds|
+--------------+--------------------+
|       5S_rRNA|[ENSG00000288601,...|
+--------------+--------------------+
only showing top 1 row



In [20]:
target_agg.count()

77111

In [21]:
x.count()

46015

In [22]:
x = x.join(target_agg, on=["approvedSymbol"], how="left").cache()

In [23]:
x.count()

46015

In [24]:
x.show(5)

+--------------+-----------+--------+-----------+-----------+-----------+-------+-------+-----------------+
|approvedSymbol|     ti_uid|phasenum|      phase|maxphasenum|   maxphase|n_drugs| meshId|        targetIds|
+--------------+-----------+--------+-----------+-----------+-----------+-------+-------+-----------------+
|           A2M|A2M-D002545|       5|  Phase III|          5|  Phase III|      1|D002545|[ENSG00000175899]|
|           A2M|A2M-D007511|       5|  Phase III|          5|  Phase III|      1|D007511|[ENSG00000175899]|
|           A2M|A2M-D010003|       2|Preclinical|          2|Preclinical|      1|D010003|[ENSG00000175899]|
|           A2M|A2M-D016491|       5|  Phase III|          5|  Phase III|      1|D016491|[ENSG00000175899]|
|           A2M|A2M-D020521|       5|  Phase III|          5|  Phase III|      1|D020521|[ENSG00000175899]|
+--------------+-----------+--------+-----------+-----------+-----------+-------+-------+-----------------+
only showing top 5 rows



In [25]:
x.filter(f.col("targetIds").isNull()).count()

150

# Join with disesase index


In [26]:
disease_index_orig.show(1, truncate=False)

+------------+-------------------------------------------+---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------+---------------------------------------------+--------------------------------------------------------------------------------------------------+-------------+-------------+--------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+
|id          |code                        

In [27]:
disease_index_orig.printSchema()

root
 |-- id: string (nullable = true)
 |-- code: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- dbXRefs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- parents: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- synonyms: struct (nullable = true)
 |    |-- hasExactSynonym: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- hasRelatedSynonym: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- hasNarrowSynonym: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- hasBroadSynonym: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- obsoleteTerms: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- obsoleteXRefs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- children: array (nullable 

In [28]:
disease_index_orig.count()

38959

In [29]:
disease_index_orig.filter(f.col("id") == "MONDO_0005090").show(truncate=False)

+-------------+--------------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
df_exploded = disease_index_orig.withColumn("dbXRef", f.explode(f.col("dbXRefs"))).withColumn(
    "dbXRef", f.upper(f.col("dbXRef"))
)

# df_exploded2 = df_exploded.withColumn("obsoleteXRef", f.explode(f.col("obsoleteXRefs"))).withColumn(
#    "obsoleteXRef", f.upper(f.col("obsoleteXRef"))
# )

mesh_pattern = r"(?i)MESH:\s*([^,\s]+)"

df_mesh = df_exploded.withColumn("meshId", f.regexp_extract(f.col("dbXRef"), mesh_pattern, 1))
df_mesh = df_mesh.select("id", "meshId").filter(f.col("meshId") != "").distinct()
df_mesh.count()

6591

In [31]:
df_exploded2 = disease_index_orig.withColumn("obsoleteXRef", f.explode(f.col("obsoleteXRefs"))).withColumn(
    "obsoleteXRef", f.upper(f.col("obsoleteXRef"))
)

mesh_pattern = r"(?i)MESH:\s*([^,\s]+)"

df_mesh2 = df_exploded2.withColumn("meshId", f.regexp_extract(f.col("obsoleteXRef"), mesh_pattern, 1))
df_mesh2 = df_mesh2.select("id", "meshId").filter(f.col("meshId") != "").distinct()
df_mesh2.count()

470

In [32]:
df_mesh_all = df_mesh.unionByName(df_mesh2).distinct()
df_mesh_all.count()

6695

In [33]:
df_mesh_all.filter(f.col("id") == "MONDO_0005090").show(truncate=False)

+-------------+-------+
|id           |meshId |
+-------------+-------+
|MONDO_0005090|D012559|
+-------------+-------+



In [34]:
df_mesh_all.filter(f.col("meshId") == "D007511").show(truncate=False)

+-------------+-------+
|id           |meshId |
+-------------+-------+
|MONDO_0005053|D007511|
|EFO_0000556  |D007511|
+-------------+-------+



In [35]:
df_mesh_all.select("meshId").distinct().count()

6207

In [36]:
df_mesh_agg = df_mesh_all.groupBy("meshId").agg(f.collect_list("id").alias("diseaseIds"))

In [37]:
df_mesh_agg.count()

6207

In [38]:
df_mesh_agg.show(1)

+------+-------------+
|meshId|   diseaseIds|
+------+-------------+
|130014|[EFO_0022194]|
+------+-------------+
only showing top 1 row



In [39]:
x.count()

46015

In [40]:
x = x.join(df_mesh_agg.select("diseaseIds", "meshId"), on=["meshId"], how="left").cache()
x.count()

46015

In [41]:
x.show()

+-------+--------------+-------------+--------+-----------+-----------+-----------+-------+-----------------+--------------------+
| meshId|approvedSymbol|       ti_uid|phasenum|      phase|maxphasenum|   maxphase|n_drugs|        targetIds|          diseaseIds|
+-------+--------------+-------------+--------+-----------+-----------+-----------+-------+-----------------+--------------------+
|D002545|           A2M|  A2M-D002545|       5|  Phase III|          5|  Phase III|      1|[ENSG00000175899]|[MONDO_0005299, H...|
|D007511|           A2M|  A2M-D007511|       5|  Phase III|          5|  Phase III|      1|[ENSG00000175899]|[MONDO_0005053, E...|
|D010003|           A2M|  A2M-D010003|       2|Preclinical|          2|Preclinical|      1|[ENSG00000175899]|     [MONDO_0005178]|
|D016491|           A2M|  A2M-D016491|       5|  Phase III|          5|  Phase III|      1|[ENSG00000175899]|       [EFO_0003875]|
|D020521|           A2M|  A2M-D020521|       5|  Phase III|          5|  Phase III|

In [44]:
x.filter(f.col("diseaseIds").isNull()).count()

5955

# Final converts


In [45]:
x = x.withColumn(
    "clinicalPhase",
    f.when(f.col("maxphase") == "Preclinical", 0)
    .when(f.col("maxphase") == "Phase I", 1)
    .when(f.col("maxphase") == "Phase II", 2)
    .when(f.col("maxphase") == "Phase III", 3)
    .when(f.col("maxphase") == "Launched", 4)
    .otherwise(None),  # Or keep it as null if it doesn't match
)

In [46]:
x.count()

46015

In [47]:
x.groupBy("maxphasenum").count().show()

+-----------+-----+
|maxphasenum|count|
+-----------+-----+
|          3| 6523|
|          5| 5763|
|          6| 6861|
|          4|12516|
|          2|14352|
+-----------+-----+



In [48]:
x.groupBy("clinicalPhase").count().show()

+-------------+-----+
|clinicalPhase|count|
+-------------+-----+
|            1| 6523|
|            3| 5763|
|            4| 6861|
|            2|12516|
|            0|14352|
+-------------+-----+



In [49]:
x.printSchema()

root
 |-- meshId: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- ti_uid: string (nullable = true)
 |-- phasenum: string (nullable = true)
 |-- phase: string (nullable = true)
 |-- maxphasenum: string (nullable = true)
 |-- maxphase: string (nullable = true)
 |-- n_drugs: string (nullable = true)
 |-- targetIds: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- clinicalPhase: integer (nullable = true)



In [50]:
x.select("meshId", "approvedSymbol").distinct().count()

29481

In [51]:
result = x.groupBy("meshId", "approvedSymbol").agg(
    # 1. Get the highest clinical phase
    f.max("clinicalPhase").alias("maxClinicalPhase"),
    f.array_distinct(f.flatten(f.collect_list(f.coalesce(f.col("diseaseIds"), f.array())))).alias("diseaseIds"),
    # 3. Do the same for targetIds
    f.array_distinct(f.flatten(f.collect_list(f.coalesce(f.col("targetIds"), f.array())))).alias("targetIds"),
)

result.count()

29481

In [ ]:
result.groupBy("maxClinicalPhase").count().show()

+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|               1| 3143|
|               3| 3680|
|               4| 5329|
|               2| 8294|
|               0| 9035|
+----------------+-----+



In [ ]:
result.show()

+----------+--------------+----------------+--------------------+-----------------+
|    meshId|approvedSymbol|maxClinicalPhase|          diseaseIds|        targetIds|
+----------+--------------+----------------+--------------------+-----------------+
|C000711409|        ERVW-1|               2|                  []|[ENSG00000242950]|
|C000711409|         HTR2A|               3|                  []|[ENSG00000102468]|
|C000711409|         OPRM1|               1|                  []|[ENSG00000112038]|
|C000711409|         TGFB1|               2|                  []|[ENSG00000105329]|
|C000711409|      TNFRSF1A|               2|                  []|[ENSG00000067182]|
|   C531854|          LIPA|               4|       [EFO_0700022]|[ENSG00000107798]|
|   C535297|        IFNAR2|               4|                  []|[ENSG00000159110]|
|   C535306|          IDH2|               0|     [MONDO_0016001]|[ENSG00000182054]|
|   C535607|          CGAS|               0|[MONDO_0018866, O...|[ENSG000001

In [ ]:
result.count()

29481

In [59]:
result.write.mode("overwrite").parquet(path_to_intermediate_data_folder + "minikel_etal_data_processed.parquet")

# test


In [7]:
phrm_df = session.spark.read.parquet(path_to_intermediate_data_folder + "minikel_etal_data_processed.parquet")
phrm_df_expl = (
    phrm_df.withColumn("targetId", f.explode(f.col("targetIds")))
    .withColumn("diseaseId", f.explode(f.col("diseaseIds")))
    .select("diseaseId", "maxClinicalPhase", "targetId")
    .withColumnRenamed("maxClinicalPhase", "ClinicalPhase")
    .withColumnRenamed("targetId", "geneId")
    .filter(f.col("geneId").isNotNull())
    .filter(f.col("diseaseId").isNotNull())
    .distinct()
)
phrm_df_expl.count()

29222

In [ ]:
phrm_df.show(1)

+----------+--------------+----------------+----------+-----------------+
|    meshId|approvedSymbol|maxClinicalPhase|diseaseIds|        targetIds|
+----------+--------------+----------------+----------+-----------------+
|C000656924|         DHODH|               3|        []|[ENSG00000102967]|
+----------+--------------+----------------+----------+-----------------+
only showing top 1 row



In [8]:
phrm_df_expl_2 = phrm_df_expl.groupBy("diseaseId", "geneId").agg(
    # 1. Get the highest clinical phase
    f.max("ClinicalPhase").alias("maxClinicalPhase")
)
phrm_df_expl_2.count()

29181

In [9]:
phrm_df_expl_2 = phrm_df_expl_2.join(
    g_p_s.select("geneId", "uniqueDiseases", "uniqueTherapeuticAreas"), "geneId", "left"
).fillna({"uniqueDiseases": 0, "uniqueTherapeuticAreas": "0"})

phrm_df_expl_2.count()

29181

In [10]:
full_l2g_evidnce = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full.drop("diseaseIds"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)
evid_indirect = chemblDrugEnrichment.evidence_to_indirect_assosiations(
    full_l2g_evidnce,
    disease_index_orig,
    use_max=True,
    efo_to_remove=efo_to_remove,
).cache()
evid_indirect.count()

151704

In [11]:
evid_indirect = (
    evid_indirect.select("targetId", "diseaseId")
    .withColumnRenamed("targetId", "geneId")
    .withColumn("geneticSupport", f.lit(1))
)

In [12]:
phrm_df_expl_2 = (
    phrm_df_expl_2.join(evid_indirect, ["geneId", "diseaseId"], "left").fillna({"geneticSupport": 0}).cache()
)
phrm_df_expl_2.count()

29181

In [13]:
phrm_df_expl_2 = phrm_df_expl_2.filter(~f.col("diseaseId").isin(efo_to_remove))
phrm_df_expl_2.count()

16746

In [14]:
df = phrm_df_expl_2.toPandas()

df = df[df["maxClinicalPhase"] > 0]
df["outcome"] = df["maxClinicalPhase"].apply(lambda x: int(x) == 4).astype(int)
df_support = df[df["uniqueDiseases"] > 0]

In [15]:
df_for_regression = df[df["maxClinicalPhase"] > 0]

In [17]:
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
df_for_regression

,geneId,diseaseId,maxClinicalPhase,uniqueDiseases,uniqueTherapeuticAreas,geneticSupport,outcome
0,ENSG00000178562,MONDO_0100096,2,15,8,0,0
1,ENSG00000213809,EFO_0004192,2,2,2,0,0
2,ENSG00000125538,MONDO_0004975,4,3,2,0,1
3,ENSG00000135519,MONDO_0004975,1,1,1,0,0
4,ENSG00000198049,EFO_0006788,2,0,0,0,0
...,...,...,...,...,...,...,...
16738,ENSG00000187266,EFO_1001459,2,1,1,0,0
16740,ENSG00000010671,EFO_0008520,3,0,0,0,0
16742,ENSG00000169252,EFO_0000341,4,0,0,0,1
16743,ENSG00000162434,EFO_0004610,2,6,5,0,0


In [ ]:
model = smf.logit(
    "outcome ~ geneticSupport",
    data=df_for_regression,
).fit(disp=False)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                10565
Model:                          Logit   Df Residuals:                    10563
Method:                           MLE   Df Model:                            1
Date:                Tue, 11 Aug 2026   Pseudo R-squ.:               0.0005974
Time:                        20:56:42   Log-Likelihood:                -5771.5
converged:                       True   LL-Null:                       -5774.9
Covariance Type:            nonrobust   LLR p-value:                  0.008620
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -1.1890      0.024    -50.206      0.000      -1.235      -1.143
geneticSupport     0.2505      0.094      2.671      0.008       0.067       0.434
